# NOTEBOOK 2: Longform WormShadow, batch and anonmyization compatible
This notebook takes regions files (created in Notebook 1) and the original dat.bin footage to create:
* Segmented or continuous .jpeg and .mp4 footage of the bin files.
* Labels output footage using orginal session name or with anonymous codenames.

The jpeg series in this notebook are used to create masks of worms in Notebook 3. The anonymous footage created by this notebook can be fed to the Blind Scoring Interface for manual scoring of anonymous footage.

This notebook requires the environment "planarian_segmentation", as do the next several notebooks.

NOTE: This notebook is functionally equivalent to LF_2_WormShadow_20251211v1_LfwithAnon.ipynb. It was reconfigured and restructured for batch processing, but the output videos and so on are the same.

### BLOCK 1: Configuration and session list
Summary: Defines all global configuration flags, directory paths, session list,and validates flag combinations before batch processing begins.

In [3]:
# === BLOCK 1: Configuration and Session List ===
# 
# Summary: Defines all global configuration flags, directory paths, session list,
# and validates flag combinations before batch processing begins.

import os
import time
import cv2
import numpy as np
import glob
import gc
import pandas as pd
import random
import csv
from pathlib import Path
from datetime import datetime
import pdb

print("Packages loaded successfully.")

# ============================================================================
# SESSION LIST FOR BATCH PROCESSING
# ============================================================================

SESSIONS_TO_PROCESS = ['2025_10_17_14_18_23_trial_1_TC']


#SESSIONS_TO_PROCESS = [
#     '2025_10_14_10_47_52_trial_1_TP',  
#     '2025_10_14_11_00_58_trial_1_TP',  
#     '2025_10_14_14_35_32_trial_1_TP',  
#     '2025_10_14_14_49_36_trial_1_TP',  
#     '2025_10_17_10_34_57_trial_1_TP', 
#     '2025_10_17_10_48_37_trial_1_TP', 
#     '2025_10_17_14_45_34_trial_1_TP', 
#     '2025_10_17_14_54_47_trial_1_TP']
    
#     '2025_10_14_10_25_19_trial_1_TC', 
#     '2025_10_14_10_35_03_trial_1_TC',
#     '2025_10_14_14_13_15_trial_1_TC',
#     '2025_10_14_14_25_09_trial_1_TC',
#     '2025_10_17_10_05_03_trial_1_TC',
#     '2025_10_17_10_18_17_trial_1_TC',
#     '2025_10_17_14_18_23_trial_1_TC',
#     '2025_10_17_14_30_05_trial_1_TC',


#     '2025_10_15_10_09_28_trial_1_TC',
#     '2025_10_15_10_20_58_trial_1_TC',
#     '2025_10_15_14_16_21_trial_1_TC',
#     '2025_10_15_14_30_16_trial_1_TC',
#     '2025_10_16_09_59_15_trial_1_TC',
#     '2025_10_16_10_10_37_trial_1_TC',
#     '2025_10_16_14_12_17_trial_1_TC',
#     '2025_10_16_14_28_16_trial_1_TC'
# ]



# ============================================================================
# GLOBAL PROCESSING FLAGS (apply to all sessions in batch)
# ============================================================================

LONG_OR_SHORT = 'long'               # Options: 'long' or 'short'
ANONYMIZATION = False                   # Options: True or False
SEPARATE_CS_ON_OFF_BACKGROUNDS = True  # True: CSon/CSoff, False: single averaged

# ============================================================================
# CAMERA AND TIMING PARAMETERS
# ============================================================================

width = 2048
height = 2048
FRAME_RATE = 10
PRE_SHIFT_FRAMES = 5
CUT_DELAY_AFTER_SHOCK = 2.15
CUT_DELAY_BEFORE_LIGHT = 2.0
TP_TOTAL_FRAMES = 274  # Total frames for TP trial videos

# ============================================================================
# DIRECTORY PATHS
# ============================================================================

HOME = "/n/holylabs/gershman_lab/Users/zkelso"
NETSCRATCH = "/n/netscratch/gershman_lab/Lab/zkelso"
HOLYLABS = "/n/holylabs/gershman_lab/Users/zkelso/Raw_data"
# DLC_PROJECT_PATH = '/n/holylabs/LABS/gershman_lab/Users/zkelso/DLC_Projects/WormTracking'
DLC_PROJECT_PATH = os.path.join(NETSCRATCH,'temporary_jpgs')

# Anonymization file paths
ANONYMIZATION_DICT_PATH = '/n/holylabs/LABS/gershman_lab/Users/zkelso/Anonymization_tools/anonymization_dictionary.csv'
LOOKUP_CSV_PATH = '/n/holylabs/LABS/gershman_lab/Users/zkelso/Anonymization_tools/anonymization_lookup.csv'
EXPERIMENT_LOG = '/n/holylabs/LABS/gershman_lab/Users/zkelso/Anonymization_tools/experiment_log.csv'

# ============================================================================
# TP BLOCK-TO-TRIALS MAPPING
# ============================================================================

TP_BLOCK_TO_TRIALS = {
    1: [1, 2],
    2: [1],
    3: [2],
    4: [2],
    5: [1, 3],
    6: [1],
    7: [1, 3],
    8: [2]
}

# ============================================================================
# FLAG VALIDATION
# ============================================================================

print("\n" + "="*70)
print("BATCH CONFIGURATION VALIDATION")
print("="*70)

# Check for invalid flag combinations
if LONG_OR_SHORT == 'long' and ANONYMIZATION == True:
    raise ValueError(
        """ERROR: LONG_OR_SHORT='long' with ANONYMIZATION=True is not valid.
Do you really want to anonymize long videos? That doesn't really make sense...
... unless you know something I don't. Comment out this error raise if you want to proceed."""
    )

if LONG_OR_SHORT == 'short' and ANONYMIZATION == False:
    raise ValueError(
        """ERROR: LONG_OR_SHORT='short' with ANONYMIZATION=False is not valid.
This is not a valid combination. Check your flags."""
    )

# ============================================================================
# CONFIGURATION SUMMARY
# ============================================================================

print("\nConfiguration valid")
print("\nGlobal Settings:")
print(f"  Processing mode: {LONG_OR_SHORT.upper()}")
print(f"  Anonymization: {'ENABLED' if ANONYMIZATION else 'DISABLED'}")
print(f"  Background mode: {'CS on/off separation' if SEPARATE_CS_ON_OFF_BACKGROUNDS else 'Single averaged'}")
print(f"  Resolution: {width} x {height}")
print(f"  Frame rate: {FRAME_RATE} fps")

print(f"\nBatch Information:")
print(f"  Total sessions to process: {len(SESSIONS_TO_PROCESS)}")
print(f"  Sessions:")
for i, session in enumerate(SESSIONS_TO_PROCESS, 1):
    print(f"    {i}. {session}")

if ANONYMIZATION:
    print(f"\nAnonymization files:")
    print(f"  Dictionary: {ANONYMIZATION_DICT_PATH}")
    print(f"  Lookup CSV: {LOOKUP_CSV_PATH}")
    print(f"  Experiment log: {EXPERIMENT_LOG}")

print("\n" + "="*70)
print("Proceed to Block 2")

Packages loaded successfully.

BATCH CONFIGURATION VALIDATION

Configuration valid

Global Settings:
  Processing mode: LONG
  Anonymization: DISABLED
  Background mode: CS on/off separation
  Resolution: 2048 x 2048
  Frame rate: 10 fps

Batch Information:
  Total sessions to process: 1
  Sessions:
    1. 2025_10_17_14_18_23_trial_1_TC

Proceed to Block 2


## BLOCK 2: Define core processing functions
Defines all core video processing functions including trial calculation,video processing, and both types of background creation (CS on/off or simple averaged).

In [4]:
# === BLOCK 2: Core Processing Functions ===
#
# Summary: Defines all core video processing functions including trial calculation,
# video processing, and both types of background creation (CS on/off or simple averaged).
"""
CORE FUNCTIONS: 
update_session_context(session_name)
    Updates global variables for the current session including binfile_to_segment, 
    COORD_FOLDER, and SESSION_TYPE based on the session name.

calculate_trial_definitions_from_timestamps()
    Reads stim_extra.csv to calculate trial start/end frame ranges for ALL trials. 
    For TC sessions, uses light_on to shock_on timing; for TP sessions, uses fixed 
    274-frame windows from light_on. All trials found in CSV are processed.

get_tp_protocol_trial_labels()
    For TP sessions, reads experiment log to get Block number and returns the protocol 
    trial labels from TP_BLOCK_TO_TRIALS. Used for labeling videos in anonymization 
    lookup, not for filtering. Validates that number of labels matches number of CSV trials.

validate_trial_ranges(data_shape, trial_definitions)
    Checks that all trial frame ranges are within the video length and that start 
    frames are less than end frames. Raises ValueError if any ranges are invalid.

make_DLC_videos_from_jpegs(folder_name, SOURCE_FRAMES_DLC)
    Creates an MP4 video file from a folder of JPEG images using OpenCV VideoWriter. 
    Displays an overwriting progress bar during encoding.

create_trial_specific_backgrounds()
    Generates two background images per coordinate set by averaging frames during 
    trial periods (CSon) and inter-trial periods (CSoff). Saves backgrounds as JPG 
    files with coordinate-specific names and creates a trial_definitions.csv file.

create_simple_averaged_background()
    Creates a single averaged background image per coordinate set by averaging all 
    frames in the video. This is a faster alternative to CS on/off backgrounds when 
    trial timing is not needed.

create_backgrounds()
    Wrapper function that calls either create_trial_specific_backgrounds() or 
    create_simple_averaged_background() based on the SEPARATE_CS_ON_OFF_BACKGROUNDS flag.
"""

def update_session_context(session_name):
    """Update session-specific variables for the current session"""
    global binfile_to_segment, COORD_FOLDER, SESSION_TYPE
    
    binfile_to_segment = session_name
    #COORD_FOLDER = os.path.join(NETSCRATCH, 'Regions_files', binfile_to_segment)
    COORD_FOLDER = os.path.join(HOLYLABS, binfile_to_segment)
    
    # Detect session type from name
    if binfile_to_segment.endswith('_TC'):
        SESSION_TYPE = 'TC'
    elif binfile_to_segment.endswith('_TP'):
        SESSION_TYPE = 'TP'
    else:
        SESSION_TYPE = 'TC'
    
    return SESSION_TYPE


def calculate_trial_definitions_from_timestamps():
    """
    Calculate trial frame ranges from stim_extra.csv timestamps.
    
    For TC sessions: Uses light_on to shock_on
    For TP sessions: Uses fixed 274-frame window from light_on
    
    ALL trials found in CSV are processed (no filtering).
    
    Returns:
        dict: {trial_num: (start_frame, end_frame)} or None if calculation fails
    """
    
    print("  Calculating trial definitions from timestamps...")
    
    # Load stimulus timing data
    # stim_csv_path = os.path.join(NETSCRATCH, 'Raw_data', binfile_to_segment, 'stim_extra.csv')
    #import pdb
    #pdb.set_trace()
    stim_csv_path = os.path.join(HOME, 'Raw_data', binfile_to_segment, 'stim_extra.csv')

    
    if not os.path.exists(stim_csv_path):
        print(f"    WARNING: Stimulus file not found: {stim_csv_path}")
        return None
    
    try:
        df = pd.read_csv(stim_csv_path)
        
        # Extract trials with valid light_on timestamps
        trials = []
        for idx, row in df.iterrows():
            if not pd.isna(row.get('white_light_on')):
                trial_info = {
                    'trial_num': idx + 1,
                    'light_on': row['white_light_on']
                }
                
                # For TC sessions, we also need shock timing
                if SESSION_TYPE == 'TC' and not pd.isna(row.get('shock_on')):
                    trial_info['shock_on'] = row['shock_on']
                    trial_info['shock_off'] = row.get('shock_off', None)
                    trials.append(trial_info)
                elif SESSION_TYPE == 'TP':
                    # TP only needs light_on
                    trials.append(trial_info)
        
        if not trials:
            raise ValueError("No valid trials found in stimulus data")
        
        # Infer video start time from first trial
        first_light = trials[0]['light_on']
        first_light_frame = 23  # Empirically verified
        video_start_ms = first_light - (first_light_frame * 1000.0 / FRAME_RATE)
        
        # Calculate trial frame windows
        calculated_trials = {}
        
        if SESSION_TYPE == 'TC':
            # TC: Use light_on to shock_on
            for trial in trials:
                trial_num = trial['trial_num']
                
                light_elapsed_ms = trial['light_on'] - video_start_ms
                light_frame = int(light_elapsed_ms / 1000.0 * FRAME_RATE)
                trial_start = light_frame - PRE_SHIFT_FRAMES
                
                shock_elapsed_ms = trial['shock_on'] - video_start_ms
                shock_frame = int(shock_elapsed_ms / 1000.0 * FRAME_RATE)
                trial_end = shock_frame - 1
                
                calculated_trials[trial_num] = (trial_start, trial_end)
        
        else:  # SESSION_TYPE == 'TP'
            # TP: Fixed window from light_on
            for trial in trials:
                trial_num = trial['trial_num']
                
                light_elapsed_ms = trial['light_on'] - video_start_ms
                light_frame = int(light_elapsed_ms / 1000.0 * FRAME_RATE)
                trial_start = light_frame - PRE_SHIFT_FRAMES
                trial_end = trial_start + TP_TOTAL_FRAMES - 1
                
                calculated_trials[trial_num] = (trial_start, trial_end)
        
        print(f"    Found {len(calculated_trials)} trials")
        return calculated_trials
        
    except Exception as e:
        print(f"    ERROR calculating trial definitions: {e}")
        return None


def get_tp_protocol_trial_labels(num_csv_trials):
    """
    Get protocol trial numbers for labeling TP videos in anonymization lookup.
    Does NOT filter trials - only provides labels for anonymization.
    
    Args:
        num_csv_trials: Number of trials found in stim_extra.csv
    
    Returns:
        list: Protocol trial numbers (e.g., [1, 3] for Block 7)
        
    Raises:
        ValueError: If number of labels doesn't match number of CSV trials
    """
    
    if SESSION_TYPE != 'TP':
        # TC sessions use sequential numbering
        return list(range(1, num_csv_trials + 1))
    
    # Read experiment log to get Block number
    if not os.path.exists(EXPERIMENT_LOG):
        raise FileNotFoundError(f"Experiment log not found: {EXPERIMENT_LOG}")
    
    exp_log_df = pd.read_csv(EXPERIMENT_LOG)
    session_row = exp_log_df[exp_log_df['Data_Folder'] == binfile_to_segment]
    
    if len(session_row) == 0:
        raise ValueError(f"Session '{binfile_to_segment}' not found in experiment log")
    
    block_number = int(session_row['Block'].iloc[0])
    
    if block_number not in TP_BLOCK_TO_TRIALS:
        raise ValueError(f"Block {block_number} not found in TP_BLOCK_TO_TRIALS dictionary")
    
    protocol_trial_labels = TP_BLOCK_TO_TRIALS[block_number]
    
    # VALIDATE: Number of labels must match number of CSV trials
    if len(protocol_trial_labels) != num_csv_trials:
        raise ValueError(
            f"TRIAL LABELING MISMATCH for session '{binfile_to_segment}':\n"
            f"  Block {block_number} expects {len(protocol_trial_labels)} trials: {protocol_trial_labels}\n"
            f"  But stim_extra.csv contains {num_csv_trials} trials\n"
            f"  Check TP_BLOCK_TO_TRIALS dictionary or Block assignment in experiment log"
        )
    
    print(f"    Block {block_number}: labeling {num_csv_trials} trials as {protocol_trial_labels}")
    return protocol_trial_labels


def validate_trial_ranges(data_shape, trial_definitions):
    """Check if trial frame ranges are valid for the video length"""
    total_frames = data_shape[0]
    
    for trial_num, (start_frame, end_frame) in trial_definitions.items():
        if end_frame >= total_frames:
            raise ValueError(f"Trial {trial_num} end frame ({end_frame}) exceeds video length ({total_frames} frames)")
        if start_frame < 0:
            raise ValueError(f"Trial {trial_num} start frame ({start_frame}) is negative")
        if start_frame > end_frame:
            raise ValueError(f"Trial {trial_num} start frame ({start_frame}) > end frame ({end_frame})")


def make_DLC_videos_from_jpegs(folder_name, SOURCE_FRAMES_DLC):
    """Create MP4 from existing JPEGs in folder"""
    
    images = sorted(glob.glob(os.path.join(SOURCE_FRAMES_DLC, "*.jpeg")))
    
    if not images:
        print(f"      WARNING: No JPEG files found at {os.path.join(SOURCE_FRAMES_DLC, '*.jpeg')}")
        return
    
    frame = cv2.imread(images[0])
    height, width, layers = frame.shape
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_filename = f"{folder_name}.mp4"
    video_path = os.path.join(DLC_PROJECT_PATH, 'videos', video_filename)
    
    os.makedirs(os.path.dirname(video_path), exist_ok=True)
    
    out = cv2.VideoWriter(video_path, fourcc, 10, (width, height))
    
    total_frames = len(images)
    
    for i, image_path in enumerate(images):
        frame = cv2.imread(image_path)
        out.write(frame)
        
        # Overwriting progress bar
        if total_frames > 10:
            progress = ((i + 1) / total_frames) * 100
            print(f"\r      Creating MP4: {progress:.0f}% [{i+1}/{total_frames}]", end='', flush=True)
    
    out.release()
    if total_frames > 10:
        print()  # New line after progress bar


def create_trial_specific_backgrounds():
    """
    Create two background images per coordinate set:
    - CSon: Average of frames during trial periods
    - CSoff: Average of frames during inter-trial periods
    """
    
    print("    Creating CS on/off backgrounds...")
    
    # Load regions file
    regions_pattern = os.path.join(COORD_FOLDER, "regions*")
    regions_files = glob.glob(regions_pattern)
    
    if not regions_files:
        print(f"      ERROR: No regions file found at {regions_pattern}")
        return
    
    cropped_dims = np.load(regions_files[0])
    
    # Load full video
    bin_pattern = os.path.join(HOME, 'Raw_data', binfile_to_segment, 'dat.bin')
    bin_files = glob.glob(bin_pattern)
    
    if not bin_files:
        print(f"      ERROR: No .bin file found")
        return
    
    SOURCE_VIDEO = bin_files[0]
    
    with open(SOURCE_VIDEO, 'rb') as f:
        data_ = np.fromfile(f, dtype=np.uint8)
        
        frame_size = width * height
        complete_frames = data_.shape[0] // frame_size
        data_ = data_[:complete_frames * frame_size]
        data_ = data_.reshape(complete_frames, height, width)
    
    # Ensure the background output directory exists
    background_output_dir = os.path.join(HOME, "Calibration_images", "Full_videos")
    os.makedirs(background_output_dir, exist_ok=True)
    
    # Determine CSon and CSoff frame indices using ALL trials
    CSon_frames = []
    for trial_num, (start_frame, end_frame) in ALL_TRIAL_DEFINITIONS.items():
        CSon_frames.extend(range(start_frame, end_frame + 1))
    
    CSon_frames = sorted(set(CSon_frames))
    all_frames = set(range(data_.shape[0]))
    CSoff_frames = sorted(all_frames - set(CSon_frames))
    
    # Process each coordinate set
    for coord_idx, coords in enumerate(cropped_dims):
        # Define crop boundaries
        x_start = int(min(coords[0], coords[2]))
        x_end = int(max(coords[0], coords[2]))
        y_start = int(min(coords[1], coords[3]))
        y_end = int(max(coords[1], coords[3]))
        
        # Ensure within bounds
        x_start = max(0, x_start)
        y_start = max(0, y_start)
        x_end = min(data_.shape[2], x_end)
        y_end = min(data_.shape[1], y_end)
        
        # Create CSon background
        CSon_sum = np.zeros((y_end - y_start, x_end - x_start), dtype=np.float64)
        for frame_idx in CSon_frames:
            frame = data_[frame_idx]
            cropped = frame[y_start:y_end, x_start:x_end]
            CSon_sum += cropped.astype(np.float64)
        CSon_background = (CSon_sum / len(CSon_frames)).astype(np.uint8)
        
        # Create CSoff background
        CSoff_sum = np.zeros((y_end - y_start, x_end - x_start), dtype=np.float64)
        for frame_idx in CSoff_frames:
            frame = data_[frame_idx]
            cropped = frame[y_start:y_end, x_start:x_end]
            CSoff_sum += cropped.astype(np.float64)
        CSoff_background = (CSoff_sum / len(CSoff_frames)).astype(np.uint8)
        
        # Save backgrounds
        coord_string = f"{coords[0]}_{coords[1]}_{coords[2]}_{coords[3]}"
        
        CSon_filename = f"{binfile_to_segment}_regions_{coord_string}_CSon.jpg"
        CSoff_filename = f"{binfile_to_segment}_regions_{coord_string}_CSoff.jpg"
        
        CSon_path = os.path.join(background_output_dir, CSon_filename)
        CSoff_path = os.path.join(background_output_dir, CSoff_filename)
        
        CSon_bgr = cv2.cvtColor(CSon_background, cv2.COLOR_GRAY2BGR)
        CSoff_bgr = cv2.cvtColor(CSoff_background, cv2.COLOR_GRAY2BGR)
        
        cv2.imwrite(CSon_path, CSon_bgr)
        cv2.imwrite(CSoff_path, CSoff_bgr)
    
    # Save trial definitions as CSV
    trial_csv_path = os.path.join(HOLYLABS, binfile_to_segment, "trial_definitions.csv")
    os.makedirs(os.path.dirname(trial_csv_path), exist_ok=True)   
    trial_df = pd.DataFrame([
        {'trial_num': trial_num, 'start_frame': start_frame, 'end_frame': end_frame}
        for trial_num, (start_frame, end_frame) in ALL_TRIAL_DEFINITIONS.items()
    ])
    trial_df.to_csv(trial_csv_path, index=False)
    
    # Clean up
    del data_
    gc.collect()
    
    print(f"      Created {len(cropped_dims)} CS on/off background pairs")


def create_simple_averaged_background():
    """
    Create single averaged background from ALL frames.
    Faster alternative to CS on/off backgrounds.
    """
    
    print("    Creating simple averaged backgrounds...")
    
    # Load regions file
    regions_pattern = os.path.join(COORD_FOLDER, "regions*")
    regions_files = glob.glob(regions_pattern)
    
    if not regions_files:
        print(f"      ERROR: No regions file found at {regions_pattern}")
        return
    
    cropped_dims = np.load(regions_files[0])
    
    # Load full video
    bin_pattern = os.path.join(HOME, 'Raw_data', binfile_to_segment, '*dat.bin')
    bin_files = glob.glob(bin_pattern)
    
    if not bin_files:
        print(f"      ERROR: No .bin file found")
        return
    
    SOURCE_VIDEO = bin_files[0]
    
    with open(SOURCE_VIDEO, 'rb') as f:
        data_ = np.fromfile(f, dtype=np.uint8)
        
        frame_size = width * height
        complete_frames = data_.shape[0] // frame_size
        data_ = data_[:complete_frames * frame_size]
        data_ = data_.reshape(complete_frames, height, width)
    
    # Ensure the background output directory exists
    background_output_dir = os.path.join(HOME, "Calibration_images", "Full_videos")
    os.makedirs(background_output_dir, exist_ok=True)
    
    # Process each coordinate set
    for coord_idx, coords in enumerate(cropped_dims):
        # Define crop boundaries
        x_start = int(min(coords[0], coords[2]))
        x_end = int(max(coords[0], coords[2]))
        y_start = int(min(coords[1], coords[3]))
        y_end = int(max(coords[1], coords[3]))
        
        # Ensure within bounds
        x_start = max(0, x_start)
        y_start = max(0, y_start)
        x_end = min(data_.shape[2], x_end)
        y_end = min(data_.shape[1], y_end)
        
        # Create averaged background from all frames
        avg_sum = np.zeros((y_end - y_start, x_end - x_start), dtype=np.float64)
        
        for frame_idx in range(data_.shape[0]):
            frame = data_[frame_idx]
            cropped = frame[y_start:y_end, x_start:x_end]
            avg_sum += cropped.astype(np.float64)
            
            # Overwriting progress bar
            if (frame_idx + 1) % 100 == 0:
                progress = ((frame_idx + 1) / data_.shape[0]) * 100
                print(f"\r      Processing coord {coord_idx+1}/{len(cropped_dims)}: {progress:.0f}%", end='', flush=True)
        
        print()  # New line after progress bar
        
        avg_background = (avg_sum / data_.shape[0]).astype(np.uint8)
        
        # Save background
        coord_string = f"{coords[0]}_{coords[1]}_{coords[2]}_{coords[3]}"
        avg_filename = f"{binfile_to_segment}_regions_{coord_string}_AVG.jpg"
        avg_path = os.path.join(background_output_dir, avg_filename)
        
        avg_bgr = cv2.cvtColor(avg_background, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(avg_path, avg_bgr)
    
    # Clean up
    del data_
    gc.collect()
    
    print(f"      Created {len(cropped_dims)} averaged backgrounds")


def create_backgrounds():
    """Wrapper to call appropriate background creation function based on flag"""
    if SEPARATE_CS_ON_OFF_BACKGROUNDS:
        create_trial_specific_backgrounds()
    else:
        create_simple_averaged_background()


print("="*70)
print("CORE PROCESSING FUNCTIONS LOADED")
print("="*70)
print("  update_session_context()")
print("  calculate_trial_definitions_from_timestamps()")
print("  get_tp_protocol_trial_labels()")
print("  validate_trial_ranges()")
print("  make_DLC_videos_from_jpegs()")
print("  create_trial_specific_backgrounds()")
print("  create_simple_averaged_background()")
print("  create_backgrounds()")
print("="*70)
print("\nProceed to Block 3.")

CORE PROCESSING FUNCTIONS LOADED
  update_session_context()
  calculate_trial_definitions_from_timestamps()
  get_tp_protocol_trial_labels()
  validate_trial_ranges()
  make_DLC_videos_from_jpegs()
  create_trial_specific_backgrounds()
  create_simple_averaged_background()
  create_backgrounds()

Proceed to Block 3.


### BLOCK 3: Define anonymization functions
Defines all anonymization-related functions and initializes the anonymization system by loading word dictionaries and existing anonymous names from the lookup CSV.

In [5]:
# === BLOCK 3: Anonymization Functions ===
#
# Summary: Defines all anonymization-related functions and initializes the anonymization
# system by loading word dictionaries and existing anonymous names from the lookup CSV.

"""
ANONYMIZATION FUNCTIONS: 
load_anonymization_dictionary(dict_path)
    Loads the CSV file containing adjectives and nouns for anonymous name generation. 
    Filters out empty or None values and returns two lists of valid words.

get_all_existing_names(lookup_csv_path)
    Reads the anonymization lookup CSV and returns a set of all existing anonymous 
    video names to prevent duplicates.

check_name_with_suffix(base_name, existing_names)
    Checks if a base name already exists in the set of used names. If it does, 
    appends numeric suffixes (_2, _3, etc.) until finding an available name.

generate_anonymous_video_name(adjectives, nouns, used_names, lookup_csv_path)
    Generates a unique anonymous name in the format Video_Adj_Adj_Noun by randomly 
    selecting two adjectives and one noun. Checks for duplicates against both 
    in-memory names and the lookup CSV, adding suffixes if needed.

create_lookup_csv(lookup_csv_path)
    Creates a new anonymization lookup CSV file with the required columns if it 
    does not already exist.

update_lookup_csv_with_trial(lookup_csv_path, anonymous_name, original_file, coords, trial_num, start_frame, end_frame)
    Appends a new record to the anonymization lookup CSV linking an anonymous video 
    name to its original session, coordinates, trial number, frame range, and timestamp. 
    Extracts metadata like session date and trial type from the filename.
"""

def load_anonymization_dictionary(dict_path):
    """Load the word dictionary for anonymous name generation"""
    
    with open(dict_path) as f:
        reader = list(csv.DictReader(f))
        
        valid_adjectives = [
            row['Adjective'] for row in reader 
            if row['Adjective'] and 
               row['Adjective'].strip() != '' and
               row['Adjective'] != 'None'
        ]
        
        valid_nouns = [
            row['Noun'] for row in reader 
            if row['Noun'] and 
               row['Noun'].strip() != '' and
               row['Noun'] != 'None'
        ]
    
    return valid_adjectives, valid_nouns


def get_all_existing_names(lookup_csv_path):
    """Get all existing anonymous names from the lookup CSV"""
    existing_names = set()
    
    if os.path.exists(lookup_csv_path):
        try:
            df = pd.read_csv(lookup_csv_path)
            if 'Anonymous_Name' in df.columns and len(df) > 0:
                all_names = df['Anonymous_Name'].dropna().tolist()
                existing_names.update(all_names)
        except Exception as e:
            pass
    
    return existing_names


def check_name_with_suffix(base_name, existing_names):
    """
    Check if base_name exists, if so add suffix _2, _3, etc.
    Returns the first available name with suffix if needed
    """
    
    if base_name not in existing_names:
        return base_name
    
    suffix_num = 2
    max_suffix_attempts = 1000
    
    while suffix_num <= max_suffix_attempts:
        candidate_name = f"{base_name}_{suffix_num}"
        
        if candidate_name not in existing_names:
            return candidate_name
        
        suffix_num += 1
    
    raise ValueError(f"Could not find available suffix for {base_name} after {max_suffix_attempts} attempts")


def generate_anonymous_video_name(adjectives, nouns, used_names=set(), lookup_csv_path=None):
    """
    Generate unique Video_Adj_Adj_Noun name with duplicate protection
    Checks both in-memory used_names and the lookup CSV for duplicates
    """
    
    # Get all existing names from CSV for comprehensive duplicate checking
    if lookup_csv_path:
        csv_existing_names = get_all_existing_names(lookup_csv_path)
        all_existing_names = used_names.union(csv_existing_names)
    else:
        all_existing_names = used_names
    
    max_attempts = 1000
    for attempt in range(max_attempts):
        # Generate base name
        adj1, adj2 = random.sample(adjectives, 2)
        noun = random.choice(nouns)
        base_name = f"Video_{adj1}_{adj2}_{noun}"
        
        # Check for duplicates and add suffix if needed
        final_name = check_name_with_suffix(base_name, all_existing_names)
        
        # Add to used_names set for this session
        used_names.add(final_name)
        
        return final_name
    
    raise ValueError(f"Could not generate unique name after {max_attempts} attempts")


def create_lookup_csv(lookup_csv_path):
    """Create lookup CSV if it doesn't exist"""
    if not os.path.exists(lookup_csv_path):
        df = pd.DataFrame(columns=[
            'Anonymous_Name', 'Original_File', 'Box_Coordinates', 
            'Trial_Number', 'Trial_Frames', 'Session_Date', 'Trial_Type', 'Timestamp_Created'
        ])
        df.to_csv(lookup_csv_path, index=False)


def update_lookup_csv_with_trial(lookup_csv_path, anonymous_name, original_file, coords, trial_num, start_frame, end_frame):
    """Add new anonymization record with trial information to lookup CSV"""
    
    # Load existing lookup
    df = pd.read_csv(lookup_csv_path)
    
    # Extract metadata from filename
    parts = original_file.split('_')
    session_date = None
    trial_type = None
    
    if len(parts) >= 6:
        try:
            session_date = f"{parts[0]}_{parts[1]}_{parts[2]}"
            trial_type = parts[-1]
        except:
            pass
    
    coord_string = f"{coords[0]}_{coords[1]}_{coords[2]}_{coords[3]}"
    frame_range = f"{start_frame}-{end_frame}"
    
    new_record = {
        'Anonymous_Name': anonymous_name,
        'Original_File': original_file,
        'Box_Coordinates': coord_string,
        'Trial_Number': trial_num,
        'Trial_Frames': frame_range,
        'Session_Date': session_date,
        'Trial_Type': trial_type,
        'Timestamp_Created': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)
    df.to_csv(lookup_csv_path, index=False)


# ============================================================================
# INITIALIZE ANONYMIZATION SYSTEM
# ============================================================================

if ANONYMIZATION:
    print("="*70)
    print("ANONYMIZATION SYSTEM INITIALIZATION")
    print("="*70)
    
    adjectives, nouns = load_anonymization_dictionary(ANONYMIZATION_DICT_PATH)
    create_lookup_csv(LOOKUP_CSV_PATH)
    
    # Load existing names to avoid duplicates (persists across all sessions)
    used_names = set()
    if os.path.exists(LOOKUP_CSV_PATH):
        try:
            existing_df = pd.read_csv(LOOKUP_CSV_PATH)
            if 'Anonymous_Name' in existing_df.columns and len(existing_df) > 0:
                used_names.update(existing_df['Anonymous_Name'].dropna().tolist())
                print(f"  Loaded {len(used_names)} existing anonymous names")
            else:
                print("  Lookup CSV exists but is empty")
        except pd.errors.EmptyDataError:
            print("  Lookup CSV is empty")
        except Exception as e:
            print(f"  Warning: Error reading lookup CSV: {e}")
    
    # Show duplicate protection status
    total_possible_names = len(adjectives) * (len(adjectives) - 1) * len(nouns)
    collision_probability = len(used_names) / total_possible_names if total_possible_names > 0 else 0
    
    print(f"  Anonymization system initialized")
    print(f"  Possible unique names: {total_possible_names:,}")
    print(f"  Names already used: {len(used_names)}")
    print(f"  Collision probability: {collision_probability:.8f}")
    print("="*70)
    
else:
    print("="*70)
    print("ANONYMIZATION DISABLED")
    print("="*70)
    print("  Using coordinate-based naming")
    print("="*70)
    adjectives, nouns, used_names = [], [], set()

print("\nProceed to Block 4")

ANONYMIZATION DISABLED
  Using coordinate-based naming

Proceed to Block 4


### BLOCK 4: Main processing loop for batches
Main batch processing loop that iterates through all sessions, calculates trials, processes videos with spatial/temporal segmentation, creates backgrounds, and handles errors.

In [16]:
# === BLOCK 4: Master Batch Control Loop ===
#
# Summary: Main batch processing loop that iterates through all sessions, calculates trials,
# processes videos with spatial/temporal segmentation, creates backgrounds, and handles errors.

# ============================================================================
# BATCH INITIALIZATION
# ============================================================================

print("="*70)
print("STARTING BATCH PROCESSING")
print("="*70)
print(f"Total sessions: {len(SESSIONS_TO_PROCESS)}")
print("="*70 + "\n")

batch_start_time = time.time()

batch_results = {
    'success': [],
    'failed': [],
    'skipped': [],
    'videos_created': 0,
    'backgrounds_created': 0
}

# ============================================================================
# MAIN BATCH LOOP
# ============================================================================

for session_idx, session in enumerate(SESSIONS_TO_PROCESS):
    session_start_time = time.time()
    
    print("\n" + "="*70)
    print(f"SESSION {session_idx + 1}/{len(SESSIONS_TO_PROCESS)}: {session}")
    print("="*70)
    
    #pdb.set_trace()
    try:
        # ====================================================================
        # STEP 1: Update session context
        # ====================================================================
        
        SESSION_TYPE = update_session_context(session)
        print(f"  Type: {SESSION_TYPE}")
        
        # ====================================================================
        # STEP 2: Calculate trial definitions
        # ====================================================================
        
        ALL_TRIAL_DEFINITIONS = calculate_trial_definitions_from_timestamps()
        
        if ALL_TRIAL_DEFINITIONS is None:
            raise RuntimeError("Failed to calculate trial definitions")
        
        # ====================================================================
        # STEP 3: Get protocol trial labels (no filtering for TP)
        # ====================================================================
        
        # Process ALL trials found in CSV
        TRIAL_DEFINITIONS = ALL_TRIAL_DEFINITIONS
        
        # Get protocol trial labels for anonymization
        if LONG_OR_SHORT == 'short' and SESSION_TYPE == 'TP':
            # For TP: Get protocol labels (e.g., [1, 3] for Block 7)
            # This validates that number of labels matches number of CSV trials
            PROTOCOL_TRIAL_LABELS = get_tp_protocol_trial_labels(len(TRIAL_DEFINITIONS))
        else:
            # For TC or long-form: Use sequential numbering
            PROTOCOL_TRIAL_LABELS = list(TRIAL_DEFINITIONS.keys())
        
        print(f"  Trials to process: {len(TRIAL_DEFINITIONS)}")
        
        if len(TRIAL_DEFINITIONS) == 0:
            raise RuntimeError("No trials found in stim_extra.csv")
        
        # ====================================================================
        # STEP 4: Load coordinate regions file
        # ====================================================================
        
        regions_pattern = os.path.join(COORD_FOLDER, "regions*")
        regions_files = glob.glob(regions_pattern)
        
        if regions_files:
                print(f"Using regions file found at {regions_pattern}")
        
        if not regions_files:
            raise FileNotFoundError(f"No regions file found at {regions_pattern}")
        
        cropped_dims = np.load(regions_files[0])
        print(f"  Coordinate sets: {len(cropped_dims)}")
        
        # ====================================================================
        # STEP 5: Load full video
        # ====================================================================
        
        print("  Loading video data...")
        bin_pattern = os.path.join(HOME, 'Raw_data', binfile_to_segment, 'dat.bin')
        bin_files = glob.glob(bin_pattern)
        
        if not bin_files:
            raise FileNotFoundError(f"No .bin file found matching pattern: {bin_pattern}")
        
        SOURCE_VIDEO = bin_files[0]
        
        with open(SOURCE_VIDEO, 'rb') as f:
            data_ = np.fromfile(f, dtype=np.uint8)
            
            frame_size = width * height
            complete_frames = data_.shape[0] // frame_size
            data_ = data_[:complete_frames * frame_size]
            data_ = data_.reshape(complete_frames, height, width)
        
        print(f"  Video loaded: {data_.shape[0]} frames")
        
        # Validate trial ranges if doing short-form processing
        if LONG_OR_SHORT == 'short':
            validate_trial_ranges(data_.shape, TRIAL_DEFINITIONS)
        
        # ====================================================================
        # STEP 6: Process videos (long-form or short-form)
        # ====================================================================
        
        print(f"\n  Processing mode: {LONG_OR_SHORT.upper()}")
        session_video_count = 0
        
        if LONG_OR_SHORT == 'long':
            # ================================================================
            # LONG-FORM PROCESSING: No temporal segmentation
            # ================================================================
            
            for coord_idx, coords in enumerate(cropped_dims):
                print(f"\n    Coordinate set {coord_idx + 1}/{len(cropped_dims)}")
                
                # Create descriptive folder name
                coord_string = f"{coords[0]}_{coords[1]}_{coords[2]}_{coords[3]}"
                folder_name = f"{binfile_to_segment}_regions_{coord_string}_fullvideo"
                
                # Create output directory
                SOURCE_FRAMES_DLC = Path(os.path.join(
                    DLC_PROJECT_PATH, 'unlabeled-data', binfile_to_segment, folder_name
                ))
                os.makedirs(SOURCE_FRAMES_DLC, exist_ok=True)
                
                # Calculate crop boundaries
                x_start = int(min(coords[0], coords[2]))
                x_end = int(max(coords[0], coords[2]))
                y_start = int(min(coords[1], coords[3]))
                y_end = int(max(coords[1], coords[3]))
                
                x_start = max(0, x_start)
                y_start = max(0, y_start)
                x_end = min(data_.shape[2], x_end)
                y_end = min(data_.shape[1], y_end)
                
                # Process ALL frames
                for frame_idx in range(data_.shape[0]):
                    frame = data_[frame_idx]
                    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                    cropped = frame_bgr[y_start:y_end, x_start:x_end]
                    
                    jpeg_filename = f"{frame_idx:05d}.jpeg"
                    jpeg_path = os.path.join(SOURCE_FRAMES_DLC, jpeg_filename)
                    cv2.imwrite(jpeg_path, cropped)
                    
                    # Overwriting progress bar
                    if (frame_idx + 1) % 100 == 0 or frame_idx == data_.shape[0] - 1:
                        # If not 1799 frames to be made, cancel - .bin video is probably messed up
                        if (data_.shape[0] not in [1798, 1799, 1800]) and (ANONYMIZATION is False):
                            data_shape_error = (f"Invalid data shape: Expected 1798, 1799, or 1800, but got {data_.shape[0]}")
                            print(data_shape_error)
                            raise ValueError(f"Invalid data shape: Expected 1798, 1799, or 1800, but got {data_.shape[0]}")
                        progress = ((frame_idx + 1) / data_.shape[0]) * 100
                        print(f"\r      Saving JPEGs: {progress:.0f}% [{frame_idx+1}/{data_.shape[0]}]", end='', flush=True)
                
                print()  # New line after progress bar
                
                # Create MP4 video
                make_DLC_videos_from_jpegs(folder_name, SOURCE_FRAMES_DLC)
                session_video_count += 1
        
        else:
            # ================================================================
            # SHORT-FORM PROCESSING: Temporal segmentation
            # ================================================================
            
            for coord_idx, coords in enumerate(cropped_dims):
                print(f"\n    Coordinate set {coord_idx + 1}/{len(cropped_dims)}")
                
                # Calculate crop boundaries
                x_start = int(min(coords[0], coords[2]))
                x_end = int(max(coords[0], coords[2]))
                y_start = int(min(coords[1], coords[3]))
                y_end = int(max(coords[1], coords[3]))
                
                x_start = max(0, x_start)
                y_start = max(0, y_start)
                x_end = min(data_.shape[2], x_end)
                y_end = min(data_.shape[1], y_end)
                
                # Process each trial separately
                for csv_trial_idx, (trial_num, (start_frame, end_frame)) in enumerate(TRIAL_DEFINITIONS.items()):
                    # Get the protocol trial number for labeling in lookup CSV
                    protocol_trial_num = PROTOCOL_TRIAL_LABELS[csv_trial_idx]
                    
                    # Display both CSV trial number and protocol label
                    if SESSION_TYPE == 'TP' and protocol_trial_num != trial_num:
                        print(f"      CSV trial {trial_num} (protocol trial {protocol_trial_num}): frames {start_frame}-{end_frame}")
                    else:
                        print(f"      Trial {trial_num}: frames {start_frame}-{end_frame}")
                    
                    # Generate folder name
                    if ANONYMIZATION:
                        anonymous_name = generate_anonymous_video_name(
                            adjectives, nouns, used_names, LOOKUP_CSV_PATH
                        )
                        folder_name = anonymous_name
                        print(f"        Anonymous name: {anonymous_name}")
                    else:
                        coord_string = f"{coords[0]}_{coords[1]}_{coords[2]}_{coords[3]}"
                        folder_name = f"{binfile_to_segment}_coords_{coord_string}_Trial{protocol_trial_num}"
                    
                    # Extract frames for this trial
                    trial_frames = data_[start_frame:end_frame+1]
                    
                    # Create output folder
                    SOURCE_FRAMES_DLC = Path(os.path.join(
                        DLC_PROJECT_PATH, 'unlabeled-data', binfile_to_segment, folder_name
                    ))
                    os.makedirs(SOURCE_FRAMES_DLC, exist_ok=True)
                    
                    # Save JPEGs
                    import pdb
                    pdb.set_trace()
                    for frame_idx, frame in enumerate(trial_frames):
                        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                        cropped = frame_bgr[y_start:y_end, x_start:x_end]
                        
                        original_frame_num = start_frame + frame_idx
                        jpeg_filename = f"{original_frame_num:05d}.jpeg"
                        jpeg_path = os.path.join(SOURCE_FRAMES_DLC, jpeg_filename)
                        cv2.imwrite(jpeg_path, cropped)
                        
                    # Confirm that expected 1799 frames have been made
                    if not ANONYMIZATION:  # Not necessary for short videos
                        print("I need something here to run cell")
                        # look at jpeg_path and jpeg_filename
                        # count contents of folder
                        # Does it equal 1799? T/F print a nice statement depending
                    
                    # Update lookup CSV with PROTOCOL trial number
                    if ANONYMIZATION:
                        update_lookup_csv_with_trial(
                            LOOKUP_CSV_PATH, 
                            anonymous_name, 
                            binfile_to_segment, 
                            coords, 
                            protocol_trial_num,  # Use protocol label, not CSV trial number
                            start_frame, 
                            end_frame
                        )
                    
                    # Create MP4 video
                    make_DLC_videos_from_jpegs(folder_name, SOURCE_FRAMES_DLC)
                    session_video_count += 1
                    
                        
                                
        # ====================================================================
        # STEP 7: Create background images
        # ====================================================================
        
        print(f"\n  Creating background images...")
        create_backgrounds()
        
       # pdb.set_trace()
        if SEPARATE_CS_ON_OFF_BACKGROUNDS:
            session_background_count = len(cropped_dims) * 2  # CSon and CSoff
            print("This part worked [1]")
        else:
            session_background_count = len(cropped_dims)  # Single averaged
            print("This part worked [2]")
        
        # ====================================================================
        # STEP 8: Clean up memory
        # ====================================================================
        
        del data_
        gc.collect()
        print("This part worked [3]")
        
        # ====================================================================
        # SESSION COMPLETE
        # ====================================================================
        
        import pdb
        #pdb.set_trace()
        session_duration = time.time() - session_start_time
        
        print(f"\n  Session complete: {session_duration/60:.1f} minutes")
        print(f"    Videos created: {session_video_count}")
        print(f"    Backgrounds created: {session_background_count}")
        
        batch_results['success'].append(session)
        batch_results['videos_created'] += session_video_count
        batch_results['backgrounds_created'] += session_background_count
        
        # Progress tracking
        elapsed_total = time.time() - batch_start_time
        avg_time_per_session = elapsed_total / (session_idx + 1)
        remaining_sessions = len(SESSIONS_TO_PROCESS) - (session_idx + 1)
        estimated_remaining = avg_time_per_session * remaining_sessions
        
        print(f"\n  Batch progress: {session_idx + 1}/{len(SESSIONS_TO_PROCESS)} sessions complete")
        print(f"  Estimated time remaining: {estimated_remaining/60:.1f} minutes")
        
    except FileNotFoundError as e:
        print(f"\n  ERROR: Missing required files")
        print(f"    {e}")
        batch_results['skipped'].append((session, str(e)))
        
    except Exception as e:
        print("This is the one that keeps popping up.")
        #pdb.set_trace()
        
        # Ask about indexERROR: Processing failed
        # index 1218 is out of bounds for axis 0 with size 1020
        print(f"\n  ERROR: Processing failed")
        print(f"    {e}")
        batch_results['failed'].append((session, str(e)))
        
    finally:
        # Always clean up memory, even if there was an error
        if 'data_' in locals():
            del data_
        gc.collect()

print("\n" + "="*70)
print("BATCH PROCESSING COMPLETE")
print("="*70)
print("\nProceed to Block 5 for summary report")

STARTING BATCH PROCESSING
Total sessions: 1


SESSION 1/1: 2025_10_17_14_18_23_trial_1_TC
  Type: TC
  Calculating trial definitions from timestamps...
    Found 3 trials
  Trials to process: 3
Using regions file found at /n/holylabs/gershman_lab/Users/zkelso/Raw_data/2025_10_17_14_18_23_trial_1_TC/regions*
  Coordinate sets: 6
  Loading video data...
  Video loaded: 1799 frames

  Processing mode: LONG

    Coordinate set 1/6
      Saving JPEGs: 56% [1000/1799]

KeyboardInterrupt: 

## BLOCK 5: Summary report
Generates comprehensive summary of batch processing results including timing, success/failure counts, and detailed error logs for any failed sessions.

In [ ]:
# === BLOCK 5: Summary Report ===
#
# Summary: Generates comprehensive summary of batch processing results including timing,
# success/failure counts, and detailed error logs for any failed sessions.

print("\n" + "="*70)
print("BATCH PROCESSING SUMMARY REPORT")
print("="*70)

batch_total_time = time.time() - batch_start_time

# ============================================================================
# TIMING SUMMARY
# ============================================================================

print("\nTiming:")
print(f"  Total elapsed time: {batch_total_time/60:.1f} minutes ({batch_total_time/3600:.2f} hours)")
print(f"  Average time per session: {(batch_total_time/len(SESSIONS_TO_PROCESS))/60:.1f} minutes")

# ============================================================================
# SESSION SUMMARY
# ============================================================================

total_sessions = len(SESSIONS_TO_PROCESS)
successful_sessions = len(batch_results['success'])
failed_sessions = len(batch_results['failed'])
skipped_sessions = len(batch_results['skipped'])

print("\nSessions:")
print(f"  Total sessions: {total_sessions}")
print(f"  Successful: {successful_sessions}")
print(f"  Failed: {failed_sessions}")
print(f"  Skipped: {skipped_sessions}")

# ============================================================================
# OUTPUT SUMMARY
# ============================================================================

print("\nOutputs created:")
print(f"  Videos: {batch_results['videos_created']}")
print(f"  Background images: {batch_results['backgrounds_created']}")

# ============================================================================
# SUCCESSFUL SESSIONS
# ============================================================================

if batch_results['success']:
    print("\nSuccessful sessions:")
    for session in batch_results['success']:
        print(f"  - {session}")

# ============================================================================
# SKIPPED SESSIONS
# ============================================================================

if batch_results['skipped']:
    print("\nSkipped sessions (missing files):")
    for session, error in batch_results['skipped']:
        print(f"  - {session}")
        print(f"      Reason: {error}")

# ============================================================================
# FAILED SESSIONS
# ============================================================================

#pdb.set_trace()
if batch_results['failed']:
    print("\nFailed sessions (errors during processing):")
    for session, error in batch_results['failed']:
        print(f"  - {session}")
        print(f"      Error: {error}")

# ============================================================================
# OUTPUT LOCATIONS
# ============================================================================

print("\nOutput locations:")
print(f"  Videos: {DLC_PROJECT_PATH}/videos/")
print(f"  JPEGs: {DLC_PROJECT_PATH}/unlabeled-data/")
print(f"  Backgrounds: {HOME}/Calibration_images/Full_videos/")

if ANONYMIZATION:
    print(f"  Lookup CSV: {LOOKUP_CSV_PATH}")

# ============================================================================
# FINAL STATUS
# ============================================================================

print("\n" + "="*70)

if failed_sessions == 0 and skipped_sessions == 0:
    print("BATCH PROCESSING COMPLETED SUCCESSFULLY")
    print("All sessions processed without errors.")
elif failed_sessions > 0:
    print("BATCH PROCESSING COMPLETED WITH ERRORS")
    print(f"{failed_sessions} session(s) failed - see error log above.")
else:
    print("BATCH PROCESSING COMPLETED WITH WARNINGS")
    print(f"{skipped_sessions} session(s) skipped - see log above.")

print("="*70)